# 🌍 CO₂ & GHG Emissions — Complete Analysis Notebook
### Environmental Economics Assignment | Our World in Data Dataset

This notebook covers **all major column groups** from the dataset:
- 📌 **Core CO₂ Emissions** — totals, per capita, growth, cumulative
- 💰 **Economy** — GDP, carbon intensity, consumption vs production, trade
- ⚡ **Natural Resources & Fuel Mix** — coal, oil, gas, flaring, cement, land-use
- 🌡️ **Policy & Climate Impact** — temperature attribution, GHG, methane, equity

---
### ⚙️ Setup — run this once in terminal before opening the notebook
```bash
pip install pandas numpy matplotlib seaborn plotly kaleido
```
Place `owid-co2-data.csv` in the **same folder** as this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.renderers.default = "notebook"
    HAS_PLOTLY = True
    print("✓ Plotly available — Plot 7 will be an interactive choropleth map")
except ImportError:
    HAS_PLOTLY = False
    print("⚠ Plotly not found — install with: pip install plotly kaleido")
    print("  Plot 7 will fall back to a ranked bar chart")

%matplotlib inline
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
    "axes.labelsize":    12,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        120,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
    "legend.framealpha": 0.85,
})

# ── Color palette for major countries ─────────────────────────────────────────
COUNTRY_COLORS = {
    "United States":       "#E63946",
    "China":               "#F4A261",
    "India":               "#2A9D8F",
    "Brazil":              "#457B9D",
    "Russia":              "#8338EC",
    "Germany":             "#06D6A0",
    "United Kingdom":      "#FFB703",
    "Japan":               "#FB8500",
    "European Union (27)": "#3A86FF",
    "Indonesia":           "#FF006E",
}

# Output folder
OUT = Path("co2_plots")
OUT.mkdir(exist_ok=True)
print(f"\n✓ All plots will be saved to: {OUT.resolve()}")

## 📥 Load & Clean Dataset

In [ ]:
# ── UPDATE THIS PATH if the CSV is in a different location ────────────────────
CSV_PATH = "owid-co2-data.csv"

df = pd.read_csv(CSV_PATH, low_memory=False)

# GDP per capita is not pre-computed — derive it
if "gdp_per_capita" not in df.columns:
    df["gdp_per_capita"] = df["gdp"] / df["population"]

# country_df  → only real countries (ISO code present), no continental aggregates
# world_df    → the pre-computed "World" aggregate row (used for global totals)
country_df = df[df["iso_code"].notna() & (df["iso_code"] != "")].copy()
world_df   = df[df["country"] == "World"].copy()

print(f"Full dataset  : {df.shape[0]:,} rows | {df['year'].min()}–{df['year'].max()}")
print(f"Country rows  : {country_df.shape[0]:,} rows | {country_df['country'].nunique()} countries")
print(f"World rows    : {len(world_df)}")
print(f"\nColumns available: {list(df.columns)}")

---
## 📌 SECTION A — Core CO₂ Emissions
Columns used: `co2`, `co2_per_capita`, `co2_per_gdp`, `co2_growth_abs`, `co2_growth_prct`, `co2_including_luc`, `cumulative_co2`

### Plot A1 — GDP per Capita vs CO₂ per Capita *(Scatter — Environmental Kuznets Curve)*
**Purpose:** Tests whether wealthier countries emit more CO₂ per person.  
**Columns:** `gdp_per_capita`, `co2_per_capita`, `population`  
**Interpretation:** Bubble size = population. Log-log axes reveal the near-linear relationship across income levels. European nations sit below the trend (better policy/energy mix); Gulf states and Australia sit above it (fossil-fuel dependence).

In [ ]:
df_v = country_df.dropna(subset=["gdp_per_capita","co2_per_capita"])
df_v = df_v[(df_v["gdp_per_capita"] > 0) & (df_v["co2_per_capita"] > 0)]
yr   = df_v["year"].max()
d    = df_v[df_v["year"] == yr][["country","gdp_per_capita","co2_per_capita","population"]].copy()

fig, ax = plt.subplots(figsize=(12, 7))
sizes   = np.clip(d["population"] / 1e6 * 0.35, 8, 550)
ax.scatter(d["gdp_per_capita"], d["co2_per_capita"],
           s=sizes, alpha=0.45, color="#4A90D9", edgecolors="white", linewidths=0.4,
           label="All countries (bubble ∝ population)")

for country, colour in COUNTRY_COLORS.items():
    row = d[d["country"] == country]
    if row.empty: continue
    ax.scatter(row["gdp_per_capita"], row["co2_per_capita"],
               s=220, color=colour, edgecolors="white", linewidths=1.3, zorder=6)
    ax.annotate(country,
                (row["gdp_per_capita"].values[0], row["co2_per_capita"].values[0]),
                xytext=(6, 3), textcoords="offset points",
                fontsize=8.5, color=colour, fontweight="bold")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("GDP per Capita (USD, log scale)")
ax.set_ylabel("CO₂ per Capita (tonnes, log scale)")
ax.set_title(f"Plot A1 — GDP per Capita vs CO₂ per Capita ({yr})\nEnvironmental Kuznets Curve | Bubble size ∝ population")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(handles=[mpatches.Patch(color=c, label=n) for n, c in COUNTRY_COLORS.items()],
          title="Highlighted countries", loc="upper left", fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig(OUT / "A1_gdp_vs_co2_per_capita.png")
plt.show()
print(f"✓ Saved")

### Plot A2 — Annual CO₂ Growth Rate over Time *(Dual-axis: Absolute + % Change)*
**Purpose:** Shows acceleration or deceleration of emissions year-on-year.  
**Columns:** `co2_growth_abs`, `co2_growth_prct`  
**Interpretation:** Negative bars = emissions fell that year (recessions, policy shocks). China's rapid growth in 2000s is visible; the 2020 COVID dip stands out.

In [ ]:
countries_g = ["United States","China","India","Germany","United Kingdom"]
palette_g   = [COUNTRY_COLORS[c] for c in countries_g]

fig, axes = plt.subplots(len(countries_g), 2, figsize=(16, 14), sharey="col")
fig.suptitle("Plot A2 — Annual CO₂ Growth: Absolute (Mt) & Percentage Change", fontsize=15, fontweight="bold", y=1.01)

for i, (country, colour) in enumerate(zip(countries_g, palette_g)):
    sub = country_df[(country_df["country"] == country) & (country_df["year"] >= 1990)].copy()
    sub = sub.dropna(subset=["co2_growth_abs","co2_growth_prct"])

    # Absolute change
    ax1 = axes[i, 0]
    colors_abs = [colour if v >= 0 else "#AAAAAA" for v in sub["co2_growth_abs"]]
    ax1.bar(sub["year"], sub["co2_growth_abs"], color=colors_abs, width=0.8, alpha=0.85)
    ax1.axhline(0, color="black", linewidth=0.7)
    ax1.set_title(f"{country} — Absolute Change (Mt)", fontsize=10)
    ax1.set_ylabel("Mt CO₂")

    # % change
    ax2 = axes[i, 1]
    colors_pct = [colour if v >= 0 else "#AAAAAA" for v in sub["co2_growth_prct"]]
    ax2.bar(sub["year"], sub["co2_growth_prct"], color=colors_pct, width=0.8, alpha=0.85)
    ax2.axhline(0, color="black", linewidth=0.7)
    ax2.set_title(f"{country} — % Change", fontsize=10)
    ax2.set_ylabel("% YoY")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=30)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "A2_co2_growth_rates.png")
plt.show()
print("✓ Saved")

### Plot A3 — CO₂ vs CO₂ Including Land-Use Change *(Line — Resource-rich countries)*
**Purpose:** Compares standard CO₂ to the fuller measure that includes deforestation.  
**Columns:** `co2`, `co2_including_luc`  
**Interpretation:** For countries like Brazil, Indonesia the gap is enormous — official CO₂ figures massively *understate* their true contribution when land is cleared.

In [ ]:
countries_luc = ["Brazil","Indonesia","United States","China","India","Democratic Republic of Congo"]
colours_luc   = ["#2D6A4F","#D62828","#E63946","#F4A261","#2A9D8F","#7B2FBE"]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("Plot A3 — CO₂ vs CO₂ Including Land-Use Change\nGap = hidden deforestation emissions", fontsize=14, fontweight="bold")

for ax, country, colour in zip(axes.flat, countries_luc, colours_luc):
    sub = country_df[(country_df["country"] == country) & (country_df["year"] >= 1950)].dropna(subset=["co2","co2_including_luc"])
    ax.fill_between(sub["year"], sub["co2"], sub["co2_including_luc"],
                    alpha=0.30, color=colour, label="Land-use gap")
    ax.plot(sub["year"], sub["co2"],              color=colour,    linewidth=2.0, label="CO₂ (excl. LUC)")
    ax.plot(sub["year"], sub["co2_including_luc"], color=colour,    linewidth=2.0, linestyle="--", label="CO₂ incl. LUC")
    ax.set_title(country, fontsize=11, fontweight="bold")
    ax.set_ylabel("Mt CO₂")
    ax.legend(fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "A3_co2_vs_co2_including_luc.png")
plt.show()
print("✓ Saved")

### Plot A4 — Cumulative CO₂ Emissions by Country *(Stacked area — Historical Responsibility)*
**Purpose:** Shows the all-time total CO₂ each country has contributed since 1750.  
**Column:** `cumulative_co2`, `share_global_cumulative_co2`  
**Interpretation:** The US has the largest historical share (~23.5%). China is closing the gap fast. Historical cumulative emissions directly drive present-day warming.

In [ ]:
top_countries = ["United States","China","Russia","Germany","United Kingdom",
                 "Japan","India","France","Canada","Poland"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle("Plot A4 — Cumulative CO₂ Emissions & Global Share (2024)", fontsize=14, fontweight="bold")

# Left: bar chart of absolute cumulative
vals = []
for c in top_countries:
    row = country_df[(country_df["country"]==c) & (country_df["year"]==2024)]["cumulative_co2"]
    vals.append(row.values[0] if not row.empty else 0)
colours_cum = plt.cm.get_cmap("tab10", len(top_countries))
bars = ax1.barh(top_countries[::-1], [v/1000 for v in vals[::-1]],
                color=[colours_cum(i) for i in range(len(top_countries))], alpha=0.85)
ax1.set_xlabel("Cumulative CO₂ (Gt)")
ax1.set_title("Absolute Cumulative Emissions (Gt)")
for bar, val in zip(bars, [v/1000 for v in vals[::-1]]):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.0f} Gt",
             va="center", fontsize=8)

# Right: share of global cumulative — pie / donut
share_vals = []
for c in top_countries:
    row = country_df[(country_df["country"]==c) & (country_df["year"]==2024)]["share_global_cumulative_co2"]
    share_vals.append(row.values[0] if not row.empty else 0)
rest = 100 - sum(share_vals)
labels_pie = top_countries + ["Rest of World"]
vals_pie   = share_vals + [rest]
explode    = [0.03]*len(labels_pie)
wedge_colours = [colours_cum(i) for i in range(len(top_countries))] + ["#DDDDDD"]
wedges, texts, autotexts = ax2.pie(
    vals_pie, labels=labels_pie, autopct=lambda p: f"{p:.1f}%" if p > 2 else "",
    startangle=140, colors=wedge_colours, explode=explode,
    pctdistance=0.82, labeldistance=1.07)
for t in texts: t.set_fontsize(8)
for t in autotexts: t.set_fontsize(7.5)
ax2.set_title("Share of Global Cumulative CO₂ (%)")

plt.tight_layout()
plt.savefig(OUT / "A4_cumulative_co2.png")
plt.show()
print("✓ Saved")

---
## 💰 SECTION B — Economy
Columns used: `gdp`, `co2_per_gdp`, `consumption_co2`, `consumption_co2_per_gdp`, `trade_co2`, `trade_co2_share`

### Plot B1 — CO₂ per GDP over Time *(Carbon Intensity of Economies)*
**Purpose:** How much CO₂ is emitted per dollar of economic output — a measure of decarbonisation progress.  
**Column:** `co2_per_gdp`  
**Interpretation:** All economies have improved. But efficiency gains ≠ absolute cuts (the rebound effect). China's dramatic fall from 2000 reflects rapid industrialisation upgrading, not clean energy.

In [ ]:
countries_b1 = ["United States","China","India","Germany","United Kingdom","Japan","Brazil","Russia"]
palette_b1   = sns.color_palette("tab10", len(countries_b1))

fig, ax = plt.subplots(figsize=(13, 6))
for country, colour in zip(countries_b1, palette_b1):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1970)]
           [["year","co2_per_gdp"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["co2_per_gdp"], label=country, color=colour, linewidth=2.2)
    last = sub.iloc[-1]
    ax.annotate(f"  {country}", (last["year"], last["co2_per_gdp"]),
                fontsize=7.5, color=colour, va="center")

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ per GDP (kg CO₂ per constant $)")
ax.set_title("Plot B1 — Carbon Intensity of GDP over Time\n(Lower = more carbon-efficient economy)")
ax.legend(fontsize=9, ncol=2, loc="upper right")
ax.set_xlim(1970, country_df["year"].max() + 5)

plt.tight_layout()
plt.savefig(OUT / "B1_co2_per_gdp_over_time.png")
plt.show()
print("✓ Saved")

### Plot B2 — Production vs Consumption vs Trade CO₂ *(Grouped Bar — Carbon Leakage)*
**Purpose:** Reveals whether a country's real carbon footprint is hidden by importing manufactured goods.  
**Columns:** `co2`, `consumption_co2`, `trade_co2`  
**Interpretation:** UK, Germany, USA: consumption CO₂ > production CO₂ → they import emissions. China, India: production > consumption → they export emissions embedded in manufactured goods. `trade_co2` positive = net importer of carbon.

In [ ]:
economies_b2 = ["United States","China","India","Germany","United Kingdom","Japan","Russia","Brazil","France","South Korea"]
rows_b2 = []
for e in economies_b2:
    sub = country_df[country_df["country"] == e].sort_values("year", ascending=False)
    for _, row in sub.iterrows():
        if pd.notna(row["co2"]) and pd.notna(row["consumption_co2"]) and pd.notna(row["trade_co2"]):
            rows_b2.append({"country": e, "Production CO₂": row["co2"],
                            "Consumption CO₂": row["consumption_co2"],
                            "Trade CO₂": row["trade_co2"], "year": row["year"]})
            break

df_b2 = pd.DataFrame(rows_b2)
x     = np.arange(len(df_b2))
w     = 0.26

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(15, 11))
fig.suptitle("Plot B2 — Production vs Consumption vs Trade CO₂\nCarbon Leakage across Major Economies", fontsize=14, fontweight="bold")

# Top: grouped bars
ax_top.bar(x - w, df_b2["Production CO₂"],  w, label="Production CO₂",  color="#E63946", alpha=0.85)
ax_top.bar(x,     df_b2["Consumption CO₂"], w, label="Consumption CO₂", color="#457B9D", alpha=0.85)
ax_top.set_xticks(x)
ax_top.set_xticklabels(df_b2["country"], rotation=20, ha="right", fontsize=10)
ax_top.set_ylabel("CO₂ Emissions (Mt)")
ax_top.set_title("Production vs Consumption CO₂ (most recent year with complete data)")
ax_top.legend()
ax_top.spines["top"].set_visible(False); ax_top.spines["right"].set_visible(False)

# Bottom: Trade CO₂ (net import/export of embedded carbon)
trade_colours = ["#E63946" if v >= 0 else "#2A9D8F" for v in df_b2["Trade CO₂"]]
ax_bot.bar(x, df_b2["Trade CO₂"], color=trade_colours, alpha=0.85, width=0.5)
ax_bot.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(df_b2["country"], rotation=20, ha="right", fontsize=10)
ax_bot.set_ylabel("Trade CO₂ (Mt)")
ax_bot.set_title("Trade CO₂ — Red = net carbon importer | Green = net carbon exporter")
ax_bot.spines["top"].set_visible(False); ax_bot.spines["right"].set_visible(False)

# Annotations
for i, (val, pct) in enumerate(zip(df_b2["Trade CO₂"], df_b2.get("trade_co2_share", [None]*len(df_b2)))):
    ax_bot.text(i, val + (15 if val >= 0 else -15), f"{val:+.0f} Mt",
                ha="center", va="bottom" if val >= 0 else "top", fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "B2_production_vs_consumption_trade_co2.png")
plt.show()
print("✓ Saved")

### Plot B3 — Trade CO₂ Share over Time *(Line — Trend in Carbon Outsourcing)*
**Purpose:** Tracks whether rich countries are increasingly outsourcing their emissions over decades.  
**Column:** `trade_co2_share` — trade CO₂ as % of total production CO₂  
**Interpretation:** Rising positive % for UK/Germany = growing share of their real footprint is made elsewhere. This is the "outsourcing of pollution" effect central to climate economics.

In [ ]:
countries_b3 = ["United Kingdom","Germany","United States","France","Japan","China","India"]
palette_b3   = sns.color_palette("Set1", len(countries_b3))

fig, ax = plt.subplots(figsize=(13, 6))
for country, colour in zip(countries_b3, palette_b3):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1990)]
           [["year","trade_co2_share"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["trade_co2_share"], label=country, color=colour, linewidth=2.2)

ax.axhline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.7)
ax.fill_between(ax.get_xlim(), 0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 100,
                alpha=0.04, color="red", label="_nolegend_")
ax.set_xlabel("Year")
ax.set_ylabel("Trade CO₂ as % of Production CO₂")
ax.set_title("Plot B3 — Trade CO₂ Share over Time\nPositive = net carbon importer | Negative = net carbon exporter")
ax.legend(fontsize=9)
ax.set_xlim(1990, country_df["year"].max())

plt.tight_layout()
plt.savefig(OUT / "B3_trade_co2_share_over_time.png")
plt.show()
print("✓ Saved")

---
## ⚡ SECTION C — Natural Resources & Fuel Mix
Columns used: `coal_co2`, `oil_co2`, `gas_co2`, `cement_co2`, `flaring_co2`, `land_use_change_co2`, `primary_energy_consumption`, `co2_per_unit_energy`

### Plot C1 — Fuel-Source Breakdown for Top 10 Emitters *(Stacked Bar)*
**Purpose:** What type of fuel drives each major emitter?  
**Columns:** `coal_co2`, `oil_co2`, `gas_co2`, `cement_co2`, `flaring_co2`  
**Interpretation:** Coal-heavy = China, India, South Africa (industrialising, energy-intensive). Oil-heavy = USA, Saudi Arabia (transport + petrochemicals). Gas-heavy = Russia, Iran (large gas reserves). Cement = proxy for construction boom.

In [ ]:
fuels_c1   = ["coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2"]
colours_c1 = ["#264653","#E9C46A","#F4A261","#E76F51","#A8DADC"]
labels_c1  = ["Coal","Oil","Gas","Cement","Flaring"]

df_c1_all = country_df.dropna(subset=fuels_c1)
latest_c1 = df_c1_all["year"].max()
df_c1     = df_c1_all[df_c1_all["year"] == latest_c1][["country"] + fuels_c1].copy()
df_c1["total"] = df_c1[fuels_c1].sum(axis=1)
top10_c1   = df_c1.nlargest(10, "total").reset_index(drop=True)

fig, (ax_main, ax_pct) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f"Plot C1 — Fuel-Source CO₂ Breakdown — Top 10 Emitters ({latest_c1})", fontsize=14, fontweight="bold")

# Absolute stacked bar
bottom = np.zeros(len(top10_c1))
for fuel, colour, label in zip(fuels_c1, colours_c1, labels_c1):
    vals = top10_c1[fuel].fillna(0).values
    ax_main.bar(top10_c1["country"], vals, bottom=bottom, color=colour, label=label, alpha=0.92)
    bottom += vals
ax_main.set_xticklabels(top10_c1["country"], rotation=25, ha="right")
ax_main.set_ylabel("CO₂ Emissions (Mt)")
ax_main.set_title("Absolute (Mt CO₂)")
ax_main.legend(title="Fuel source", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax_main.spines["top"].set_visible(False); ax_main.spines["right"].set_visible(False)

# 100% stacked bar (percentage share)
df_pct = top10_c1[fuels_c1].div(top10_c1["total"], axis=0) * 100
bottom_pct = np.zeros(len(df_pct))
for fuel, colour, label in zip(fuels_c1, colours_c1, labels_c1):
    vals = df_pct[fuel].fillna(0).values
    ax_pct.bar(top10_c1["country"], vals, bottom=bottom_pct, color=colour, label=label, alpha=0.92)
    bottom_pct += vals
ax_pct.set_xticklabels(top10_c1["country"], rotation=25, ha="right")
ax_pct.set_ylabel("Share of Emissions (%)")
ax_pct.set_title("Proportional Mix (%)")
ax_pct.set_ylim(0, 100)
ax_pct.spines["top"].set_visible(False); ax_pct.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "C1_fuel_breakdown_stacked.png")
plt.show()
print("✓ Saved")

### Plot C2 — Flaring CO₂ *(Bar — Oil Extraction Waste)*
**Purpose:** Gas flaring is a direct indicator of inefficient oil extraction practices.  
**Column:** `flaring_co2`  
**Interpretation:** Russia, USA, Iraq and Iran top the flaring list. High flaring = regulatory failures in the oil sector. Targeted policy (methane regulations, flaring bans) can cut this quickly.

In [ ]:
df_flare = country_df.dropna(subset=["flaring_co2"])
latest_fl = df_flare["year"].max()
top_flare = (df_flare[df_flare["year"] == latest_fl][["country","flaring_co2"]]
             .nlargest(15,"flaring_co2").reset_index(drop=True))

fig, ax = plt.subplots(figsize=(13, 6))
colours_fl = plt.cm.get_cmap("OrRd", len(top_flare))
bars = ax.bar(top_flare["country"], top_flare["flaring_co2"],
              color=[colours_fl(i/len(top_flare)) for i in range(len(top_flare))],
              edgecolor="white", linewidth=0.5, alpha=0.9)
for bar, val in zip(bars, top_flare["flaring_co2"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8)

ax.set_xticklabels(top_flare["country"], rotation=30, ha="right")
ax.set_ylabel("Flaring CO₂ (Mt)")
ax.set_title(f"Plot C2 — Top 15 Countries by Gas Flaring Emissions ({latest_fl})\nFlaring = waste from oil extraction — an indicator of regulatory failure")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "C2_flaring_co2.png")
plt.show()
print("✓ Saved")

### Plot C3 — Land-Use Change CO₂ *(Line + Fill — Deforestation Emissions)*
**Purpose:** Tracks CO₂ from forest clearing and land conversion.  
**Column:** `land_use_change_co2`  
**Interpretation:** Brazil's peak in the 1990s–2000s matches Amazon agricultural expansion. The drop after 2004 shows the PPCDAM policy worked. Indonesia surged in the 2000s (palm oil). DRC is rising — the frontier of future deforestation risk.

In [ ]:
countries_c3 = ["Brazil","Indonesia","Democratic Republic of Congo","Russia","India","United States"]
colours_c3   = ["#2D6A4F","#D62828","#7B2FBE","#8338EC","#2A9D8F","#E63946"]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("Plot C3 — Land-Use Change CO₂ Emissions\nDeforestation & land conversion across key countries", fontsize=14, fontweight="bold")

for ax, country, colour in zip(axes.flat, countries_c3, colours_c3):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1850)]
           [["year","land_use_change_co2"]].dropna())
    if sub.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(country, fontsize=10); continue
    ax.plot(sub["year"], sub["land_use_change_co2"], color=colour, linewidth=2.0)
    ax.fill_between(sub["year"], sub["land_use_change_co2"], 0,
                    where=(sub["land_use_change_co2"] > 0), alpha=0.20, color=colour, label="Positive (source)")
    ax.fill_between(sub["year"], sub["land_use_change_co2"], 0,
                    where=(sub["land_use_change_co2"] < 0), alpha=0.20, color="steelblue", label="Negative (sink)")
    ax.axhline(0, color="grey", linewidth=0.7, linestyle="--")
    ax.set_title(country, fontsize=10, fontweight="bold", color=colour)
    ax.set_ylabel("Mt CO₂"); ax.set_xlabel("Year")
    ax.legend(fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "C3_land_use_change_co2.png")
plt.show()
print("✓ Saved")

### Plot C4 — Carbon Intensity of the Energy Mix over Time *(Line)*
**Purpose:** Measures CO₂ per unit of energy consumed — how clean is the national grid?  
**Column:** `co2_per_unit_energy`  
**Interpretation:** France (nuclear) and Brazil (hydro) maintain very low intensity. Coal-heavy South Africa and India are highest. UK/Germany are declining fast due to wind/solar. This is the most direct measure of energy transition success.

In [ ]:
countries_c4 = ["United States","China","India","France","Germany","United Kingdom","Japan","Brazil","South Africa","South Korea"]
palette_c4   = sns.color_palette("tab10", len(countries_c4))

fig, ax = plt.subplots(figsize=(13, 6))
for country, colour in zip(countries_c4, palette_c4):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1970)]
           [["year","co2_per_unit_energy"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["co2_per_unit_energy"], label=country, color=colour, linewidth=2.0)
    last = sub.iloc[-1]
    ax.annotate(f"  {country}", (last["year"], last["co2_per_unit_energy"]),
                fontsize=7.5, color=colour, va="center")

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ per Unit Energy (kg CO₂ / kWh)")
ax.set_title("Plot C4 — Carbon Intensity of the Energy Mix over Time\n(Lower = cleaner energy grid)")
ax.legend(fontsize=8.5, ncol=2)
ax.set_xlim(1970, country_df["year"].max() + 8)

plt.tight_layout()
plt.savefig(OUT / "C4_co2_per_unit_energy.png")
plt.show()
print("✓ Saved")

### Plot C5 — Primary Energy Consumption vs CO₂ Emissions *(Scatter)*
**Purpose:** Links total energy use to emissions — is higher energy use always higher emissions?  
**Columns:** `primary_energy_consumption`, `co2`, `co2_per_unit_energy`  
**Interpretation:** Countries above the trend line are carbon-inefficient for their energy use. Those below have cleaner energy mixes (nuclear, hydro, renewables). The colour shows carbon intensity — a 3-variable view.

In [ ]:
df_c5 = country_df.dropna(subset=["primary_energy_consumption","co2","co2_per_unit_energy"])
yr_c5 = df_c5["year"].max()
d_c5  = df_c5[df_c5["year"] == yr_c5].copy()

fig, ax = plt.subplots(figsize=(12, 7))
sc = ax.scatter(d_c5["primary_energy_consumption"], d_c5["co2"],
                c=d_c5["co2_per_unit_energy"], cmap="YlOrRd",
                s=80, alpha=0.75, edgecolors="white", linewidths=0.4)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("CO₂ per Unit Energy (kg/kWh) — colour = grid carbon intensity")

for country in ["United States","China","India","Russia","Germany","France","Brazil","South Africa","Japan"]:
    row = d_c5[d_c5["country"] == country]
    if row.empty: continue
    col = COUNTRY_COLORS.get(country, "#333333")
    ax.scatter(row["primary_energy_consumption"], row["co2"],
               s=200, color=col, edgecolors="white", linewidths=1.3, zorder=6)
    ax.annotate(country, (row["primary_energy_consumption"].values[0], row["co2"].values[0]),
                xytext=(5, 3), textcoords="offset points", fontsize=8.5, color=col, fontweight="bold")

ax.set_xlabel("Primary Energy Consumption (TWh)")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_title(f"Plot C5 — Energy Consumption vs CO₂ Emissions ({yr_c5})\nColour = carbon intensity of energy mix")

plt.tight_layout()
plt.savefig(OUT / "C5_energy_vs_co2.png")
plt.show()
print("✓ Saved")

---
## 🌡️ SECTION D — Policy & Climate Impact
Columns used: `temperature_change_from_co2`, `temperature_change_from_ghg`, `share_of_temperature_change_from_ghg`, `share_global_co2`, `total_ghg`, `methane`, `nitrous_oxide`, `ghg_per_capita`

### Plot D1 — Global CO₂ Trend with Policy Milestones *(Line + Annotations)*
**Purpose:** The definitive macro narrative — global emissions vs. every major climate agreement.  
**Columns:** `co2` (World aggregate)  
**Interpretation:** Emissions rose through every agreement. Kyoto (1997) slowed Annex-I growth but China surged. Paris (2015) shows only a minor inflection. The chart makes a powerful argument for binding targets with carbon pricing.

In [ ]:
world9 = df[df["country"] == "World"][["year","co2"]].dropna()
if world9.empty:
    world9 = country_df.groupby("year", as_index=False)["co2"].sum()
world9 = world9[world9["year"] >= 1850].copy()

milestones = {
    1972: ("Stockholm\nConference", "top"),
    1988: ("IPCC\nFounded",         "bottom"),
    1992: ("UNFCCC\nRio",           "top"),
    1997: ("Kyoto\nProtocol",       "bottom"),
    2005: ("Kyoto\nIn Force",       "top"),
    2015: ("Paris\nAgreement",      "bottom"),
    2021: ("COP26\nGlasgow",        "top"),
}

fig, ax = plt.subplots(figsize=(15, 7))
ax.fill_between(world9["year"], world9["co2"], alpha=0.13, color="#E63946")
ax.plot(world9["year"], world9["co2"], color="#E63946", linewidth=2.5, label="Global CO₂ (Mt)")

ymax = world9["co2"].max()
for yr_m, (label, pos) in milestones.items():
    ax.axvline(yr_m, color="#555555", linewidth=1.0, linestyle="--", alpha=0.65)
    y_text = ymax * 0.94 if pos == "top" else ymax * 0.08
    ax.text(yr_m, y_text, f"{yr_m}\n{label}",
            ha="center", va="top" if pos == "top" else "bottom",
            fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="#BBBBBB", alpha=0.9))

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("CO₂ Emissions (Mt)", fontsize=12)
ax.set_title("Plot D1 — Global CO₂ Emissions Trend with Key Policy Milestones\nEmissions have risen through every major climate agreement", fontsize=14, fontweight="bold")
ax.set_xlim(1850, world9["year"].max() + 3)
ax.legend(loc="upper left", fontsize=10)

plt.tight_layout()
plt.savefig(OUT / "D1_global_co2_trend_milestones.png")
plt.show()
print("✓ Saved")

### Plot D2 — Country Contribution to Global Temperature Rise *(Bar — Climate Justice)*
**Purpose:** Which countries have caused the most actual warming, and what % of global temperature rise are they responsible for?  
**Columns:** `temperature_change_from_co2`, `temperature_change_from_ghg`, `share_of_temperature_change_from_ghg`  
**Interpretation:** USA leads on CO₂-driven warming despite China having higher current emissions — because the US burned carbon for longer. India's share is surprisingly small given its population (low historical responsibility). This is the core metric for CBDR-RC arguments.

In [ ]:
df_temp = country_df[country_df["year"] == 2024][
    ["country","temperature_change_from_co2","temperature_change_from_ghg","share_of_temperature_change_from_ghg"]
].dropna().nlargest(15, "temperature_change_from_ghg").reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle("Plot D2 — Country Contributions to Global Temperature Rise (2024)\nCumulative attribution of observed warming", fontsize=14, fontweight="bold")

cmap_t = plt.cm.get_cmap("hot_r", len(df_temp))

for ax, col, title in zip(axes,
    ["temperature_change_from_co2","temperature_change_from_ghg","share_of_temperature_change_from_ghg"],
    ["From CO₂ alone (°C)","From All GHGs (°C)","Share of Global Warming (%)"]):
    colours_t = [COUNTRY_COLORS.get(c, cmap_t(i/len(df_temp))) for i, c in enumerate(df_temp["country"])]
    bars = ax.barh(df_temp["country"][::-1], df_temp[col][::-1],
                   color=colours_t[::-1], alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, df_temp[col][::-1]):
        ax.text(val + max(df_temp[col])*0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}" if "share" not in col else f"{val:.1f}%",
                va="center", fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "D2_temperature_change_attribution.png")
plt.show()
print("✓ Saved")

### Plot D3 — Full GHG Breakdown: CO₂, Methane, Nitrous Oxide *(Stacked Bar)*
**Purpose:** CO₂ is not the only greenhouse gas — methane and N₂O are critical for agriculture-heavy economies.  
**Columns:** `co2`, `methane`, `nitrous_oxide`, `total_ghg`  
**Interpretation:** Brazil's total GHG is dominated by methane (cattle) and N₂O — its CO₂ figure alone is misleading. India's methane (rice paddies + cattle) is massive. Russia's methane comes mainly from gas pipeline leaks.

In [ ]:
countries_d3 = ["China","United States","India","Russia","Brazil","Indonesia","Japan","Germany","Canada","South Africa"]
latest_d3 = 2024

rows_d3 = []
for c in countries_d3:
    row = country_df[(country_df["country"]==c)&(country_df["year"]==latest_d3)][["co2","methane","nitrous_oxide","total_ghg"]].dropna()
    if not row.empty:
        rows_d3.append({"country": c,
                        "CO₂": row["co2"].values[0],
                        "Methane": row["methane"].values[0],
                        "Nitrous Oxide": row["nitrous_oxide"].values[0]})

df_d3 = pd.DataFrame(rows_d3)
ghg_cols = ["CO₂","Methane","Nitrous Oxide"]
ghg_colours = ["#E63946","#F4A261","#2A9D8F"]

fig, (ax_abs, ax_pct) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f"Plot D3 — Full GHG Breakdown: CO₂ + Methane + Nitrous Oxide ({latest_d3})", fontsize=14, fontweight="bold")

bottom_abs = np.zeros(len(df_d3))
for col, colour in zip(ghg_cols, ghg_colours):
    vals = df_d3[col].values
    ax_abs.bar(df_d3["country"], vals, bottom=bottom_abs, label=col, color=colour, alpha=0.88)
    bottom_abs += vals
ax_abs.set_xticklabels(df_d3["country"], rotation=25, ha="right")
ax_abs.set_ylabel("Emissions (Mt CO₂-equivalent)")
ax_abs.set_title("Absolute GHG (Mt CO₂-eq)")
ax_abs.legend(); ax_abs.spines["top"].set_visible(False); ax_abs.spines["right"].set_visible(False)

df_d3_pct = df_d3[ghg_cols].div(df_d3[ghg_cols].sum(axis=1), axis=0) * 100
bottom_pct = np.zeros(len(df_d3))
for col, colour in zip(ghg_cols, ghg_colours):
    ax_pct.bar(df_d3["country"], df_d3_pct[col].values, bottom=bottom_pct, label=col, color=colour, alpha=0.88)
    bottom_pct += df_d3_pct[col].values
ax_pct.set_xticklabels(df_d3["country"], rotation=25, ha="right")
ax_pct.set_ylabel("Share (%)")
ax_pct.set_title("GHG Mix — % Share")
ax_pct.set_ylim(0, 100)
ax_pct.legend(); ax_pct.spines["top"].set_visible(False); ax_pct.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "D3_full_ghg_breakdown.png")
plt.show()
print("✓ Saved")

### Plot D4 — Share of Global CO₂ over Time *(Area Chart — Power Shift)*
**Purpose:** Shows how each country's share of global emissions has evolved since 1900.  
**Column:** `share_global_co2`  
**Interpretation:** The US dominated until China's industrial rise from ~2000. The EU has shrunk dramatically. India is the only major economy whose share is growing. This is the "power shift" chart in climate negotiations.

In [ ]:
countries_d4 = ["United States","China","India","Russia","Germany","United Kingdom","Japan","France","Canada","Brazil"]
palette_d4   = sns.color_palette("tab10", len(countries_d4))

fig, ax = plt.subplots(figsize=(15, 7))
for country, colour in zip(countries_d4, palette_d4):
    sub = (country_df[(country_df["country"] == country) & (country_df["year"] >= 1900)]
           [["year","share_global_co2"]].dropna())
    if sub.empty: continue
    ax.plot(sub["year"], sub["share_global_co2"], label=country, color=colour, linewidth=2.0)
    last = sub.iloc[-1]
    ax.annotate(f"  {country} ({last['share_global_co2']:.1f}%)",
                (last["year"], last["share_global_co2"]),
                fontsize=7.5, color=colour, va="center")

ax.set_xlabel("Year")
ax.set_ylabel("Share of Global CO₂ (%)")
ax.set_title("Plot D4 — Share of Global CO₂ Emissions over Time\nChina surpassed the US in the 2000s; India's share is growing")
ax.legend(fontsize=8.5, ncol=2, loc="upper left")
ax.set_xlim(1900, country_df["year"].max() + 12)

plt.tight_layout()
plt.savefig(OUT / "D4_share_global_co2.png")
plt.show()
print("✓ Saved")

### Plot D5 — Historical Responsibility vs Current Per-Capita Emissions *(Scatter — Climate Justice)*
**Purpose:** The single most important chart for climate equity debates.  
**Columns:** `cumulative_co2` (x — who caused warming), `co2_per_capita` (y — current lifestyle)  
**Interpretation:** Top-right = most responsible AND still living high-carbon lifestyles (USA, Australia). Bottom-left = historically innocent AND low-carbon (most of Africa, South Asia). India: massive population, low cumulative, low per-capita — the strongest CBDR-RC case.

In [ ]:
df_d5 = country_df.dropna(subset=["cumulative_co2","co2_per_capita"])
df_d5 = df_d5[(df_d5["cumulative_co2"] > 0) & (df_d5["co2_per_capita"] > 0)]
yr_d5 = df_d5["year"].max()
d5    = df_d5[df_d5["year"] == yr_d5][["country","cumulative_co2","co2_per_capita","population"]].copy()

label_d5 = ["United States","China","Russia","Germany","United Kingdom","India","Japan",
            "Canada","Australia","Brazil","France","South Africa","South Korea","Saudi Arabia",
            "Indonesia","Nigeria","Bangladesh","Pakistan"]

fig, ax = plt.subplots(figsize=(13, 8))
sizes_d5 = np.clip(d5["population"] / 1e6 * 0.45, 8, 700)
ax.scatter(d5["cumulative_co2"], d5["co2_per_capita"],
           s=sizes_d5, alpha=0.40, color="#6B9BC3", edgecolors="white", linewidths=0.3,
           label="All countries")

for country in label_d5:
    row = d5[d5["country"] == country]
    if row.empty: continue
    col = COUNTRY_COLORS.get(country, "#444444")
    ax.scatter(row["cumulative_co2"], row["co2_per_capita"],
               s=180, color=col, edgecolors="white", linewidths=1.2, zorder=6)
    ax.annotate(country, (row["cumulative_co2"].values[0], row["co2_per_capita"].values[0]),
                xytext=(5, 3), textcoords="offset points", fontsize=8, color=col, fontweight="bold")

# Quadrant lines
med_x = d5["cumulative_co2"].median()
med_y = d5["co2_per_capita"].median()
ax.axvline(med_x, color="grey", linewidth=0.8, linestyle=":", alpha=0.7)
ax.axhline(med_y, color="grey", linewidth=0.8, linestyle=":", alpha=0.7)
ax.text(med_x * 2.5, med_y * 3.5, "High historical\nresponsibility\n+ high lifestyle", fontsize=8, color="#E63946", alpha=0.7)
ax.text(0.005, med_y * 0.1, "Low historical\nresponsibility\n+ low lifestyle", fontsize=8, color="steelblue", alpha=0.7)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Cumulative CO₂ Emissions (Mt, log) — Historical Responsibility")
ax.set_ylabel("CO₂ per Capita today (tonnes, log) — Current Lifestyle")
ax.set_title(f"Plot D5 — Historical Responsibility vs Current Per-Capita Emissions ({yr_d5})\nClimate Justice: bubble size ∝ population")

plt.tight_layout()
plt.savefig(OUT / "D5_historical_vs_current_climate_justice.png")
plt.show()
print("✓ Saved")

### Plot D6 — World Choropleth Map of CO₂ per Capita *(Geographic Distribution)*
**Purpose:** Spatial view of who emits most per person.  
**Column:** `co2_per_capita`  
**Interpretation:** Gulf states + Australia dominate per-capita rankings. Africa and South/Southeast Asia are very low. The geographic pattern maps almost perfectly onto income inequality — supporting the equity argument that climate burden must be differentiated.

In [ ]:
df_d6 = country_df.dropna(subset=["co2_per_capita"])
df_d6 = df_d6[df_d6["co2_per_capita"] > 0]
yr_d6 = df_d6["year"].max()
d6    = df_d6[df_d6["year"] == yr_d6][["country","iso_code","co2_per_capita"]].copy()

if HAS_PLOTLY:
    fig_d6 = px.choropleth(
        d6, locations="iso_code", color="co2_per_capita",
        hover_name="country",
        color_continuous_scale="YlOrRd",
        range_color=(0, d6["co2_per_capita"].quantile(0.95)),
        labels={"co2_per_capita": "CO₂ per Capita (t)"},
        title=f"Plot D6 — CO₂ per Capita by Country ({yr_d6})",
    )
    fig_d6.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=520,
                         coloraxis_colorbar=dict(title="Tonnes CO₂\nper capita"))
    fig_d6.write_image(str(OUT / "D6_choropleth_co2_per_capita.png"), width=1400, height=700, scale=2)
    fig_d6.show()
    print("✓ Saved interactive choropleth")
else:
    top30 = d6.nlargest(30, "co2_per_capita").reset_index(drop=True)
    cmap6 = plt.cm.get_cmap("YlOrRd", 30)
    fig, ax = plt.subplots(figsize=(13, 9))
    ax.barh(top30["country"][::-1], top30["co2_per_capita"][::-1],
            color=[cmap6(i/30) for i in range(len(top30))][::-1],
            edgecolor="white", linewidth=0.4)
    sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(0, top30["co2_per_capita"].max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("CO₂ per Capita (tonnes)")
    ax.set_xlabel("CO₂ per Capita (tonnes)")
    ax.set_title(f"Plot D6 — Top 30 Countries by CO₂ per Capita ({yr_d6})\n(Install plotly for interactive world map)")
    ax.spines["left"].set_visible(False); ax.tick_params(left=False)
    plt.tight_layout()
    plt.savefig(OUT / "D6_co2_per_capita_ranked.png")
    plt.show()
    print("✓ Saved (bar chart; install plotly for choropleth)")

### Plot D7 — GHG per Capita Comparison *(Heatmap — Multi-country, Multi-decade)*
**Purpose:** Compact comparison of per-capita GHG burden across countries and decades.  
**Column:** `ghg_per_capita`  
**Interpretation:** Each row is a country; each column a decade. Dark red = high burden. The heatmap reveals which countries have improved (UK, Germany) vs. which have surged (Qatar, UAE) vs. remained stubbornly high (USA, Australia).

In [ ]:
countries_d7 = ["United States","China","India","Russia","Germany","United Kingdom",
                "Japan","Brazil","Canada","Australia","South Africa","Saudi Arabia",
                "France","South Korea","Indonesia","Iran","Mexico","Nigeria"]
decades = [1990, 2000, 2010, 2015, 2020, 2024]

heatmap_data = {}
for country in countries_d7:
    row_vals = []
    for yr_h in decades:
        sub = country_df[(country_df["country"]==country) & (country_df["year"]==yr_h)]["ghg_per_capita"]
        row_vals.append(sub.values[0] if not sub.empty and not sub.isna().all() else np.nan)
    heatmap_data[country] = row_vals

df_hm = pd.DataFrame(heatmap_data, index=decades).T

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df_hm, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "GHG per Capita (t CO₂-eq)"},
            ax=ax)
ax.set_title("Plot D7 — GHG per Capita Heatmap\nAcross Major Countries & Decades (t CO₂-equivalent)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Country")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(OUT / "D7_ghg_per_capita_heatmap.png")
plt.show()
print("✓ Saved")

---
## 🔬 SECTION E — Synthesis Plots
Multi-column plots that combine insights across all four groups.

### Plot E1 — CO₂ per GDP over Time *(Carbon Decoupling Check)*
**Purpose:** Have major economies actually decoupled economic growth from emissions?  
**Method:** Overlay GDP index and CO₂ index (both rebased to 1990=100) for each country.  
**Interpretation:** A widening gap (GDP rising, CO₂ falling) = real decoupling. Relative decoupling = both rise but GDP faster. No decoupling = both rise in tandem. Germany and UK show real decoupling; China shows relative only.

In [ ]:
countries_e1 = ["United States","China","Germany","United Kingdom","India","Japan"]
palette_e1   = sns.color_palette("tab10", len(countries_e1))
BASE_YEAR    = 1990

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Plot E1 — Economic vs CO₂ Decoupling\n(Index 1990=100: GDP vs CO₂)", fontsize=14, fontweight="bold")

for ax, country, colour in zip(axes.flat, countries_e1, palette_e1):
    sub = country_df[(country_df["country"] == country) & (country_df["year"] >= BASE_YEAR)].dropna(subset=["gdp","co2"])
    if sub.empty or BASE_YEAR not in sub["year"].values:
        ax.set_title(f"{country}\n(insufficient data)"); continue
    base_gdp = sub[sub["year"]==BASE_YEAR]["gdp"].values[0]
    base_co2 = sub[sub["year"]==BASE_YEAR]["co2"].values[0]
    idx_gdp  = sub["gdp"] / base_gdp * 100
    idx_co2  = sub["co2"] / base_co2 * 100

    ax.plot(sub["year"], idx_gdp, color="#2196F3", linewidth=2.2, label="GDP index")
    ax.plot(sub["year"], idx_co2, color="#E63946", linewidth=2.2, label="CO₂ index", linestyle="--")
    ax.fill_between(sub["year"], idx_gdp, idx_co2,
                    where=(idx_gdp > idx_co2), alpha=0.12, color="green", label="Decoupled")
    ax.fill_between(sub["year"], idx_gdp, idx_co2,
                    where=(idx_gdp <= idx_co2), alpha=0.12, color="red", label="Re-coupled")
    ax.axhline(100, color="grey", linewidth=0.7, linestyle=":")
    ax.set_title(f"{country}", fontsize=11, fontweight="bold", color=colour)
    ax.set_ylabel("Index (1990=100)"); ax.legend(fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "E1_decoupling_gdp_vs_co2.png")
plt.show()
print("✓ Saved")

### Plot E2 — Methane & Nitrous Oxide vs CO₂ *(Scatter — Agricultural Emitters)*
**Purpose:** Identifies countries where non-CO₂ gases dominate — agriculture-heavy economies.  
**Columns:** `co2`, `methane`, `nitrous_oxide`  
**Interpretation:** Brazil, India, Pakistan: methane from livestock and paddy fields rivals CO₂. Any net-zero policy ignoring methane is incomplete for these countries.

In [ ]:
df_e2 = country_df[country_df["year"] == 2024][["country","co2","methane","nitrous_oxide","population"]].dropna()
df_e2["non_co2"] = df_e2["methane"] + df_e2["nitrous_oxide"]
df_e2["methane_share"] = df_e2["methane"] / (df_e2["co2"] + df_e2["non_co2"]) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Plot E2 — CO₂ vs Methane & Nitrous Oxide (2024)\nNon-CO₂ gases are critical for agriculture-heavy economies", fontsize=13, fontweight="bold")

# Left: CO2 vs Methane
sizes_e2 = np.clip(df_e2["population"] / 1e6 * 0.35, 8, 500)
sc = axes[0].scatter(df_e2["co2"], df_e2["methane"],
                     c=df_e2["methane_share"], cmap="YlOrRd",
                     s=sizes_e2, alpha=0.7, edgecolors="white", linewidths=0.4)
plt.colorbar(sc, ax=axes[0], label="Methane as % of total GHG")
for country in ["Brazil","India","China","United States","Russia","Pakistan","Indonesia","Nigeria","Argentina"]:
    row = df_e2[df_e2["country"]==country]
    if row.empty: continue
    col = COUNTRY_COLORS.get(country,"#333")
    axes[0].scatter(row["co2"], row["methane"], s=180, color=col, edgecolors="white", zorder=6)
    axes[0].annotate(country, (row["co2"].values[0], row["methane"].values[0]),
                     xytext=(5,3), textcoords="offset points", fontsize=8, color=col, fontweight="bold")
axes[0].set_xlabel("CO₂ (Mt)"); axes[0].set_ylabel("Methane (Mt CO₂-eq)")
axes[0].set_title("CO₂ vs Methane Emissions")
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

# Right: top methane % share
top_meth = df_e2.nlargest(15,"methane_share")
axes[1].barh(top_meth["country"][::-1], top_meth["methane_share"][::-1],
             color=plt.cm.get_cmap("OrRd", 15)([i/15 for i in range(15)])[::-1], alpha=0.85)
axes[1].set_xlabel("Methane as % of Total GHG")
axes[1].set_title("Countries Where Methane Dominates")
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / "E2_methane_nitrous_oxide_vs_co2.png")
plt.show()
print("✓ Saved")

---
## ✅ Complete Plot Summary

| Section | Plot | Key Column(s) | Main Takeaway |
|---------|------|---------------|---------------|
| **A. Core CO₂** | A1 — GDP vs CO₂ per capita | `gdp_per_capita`, `co2_per_capita` | Wealthier countries emit more; Europe beats the trend |
| | A2 — Annual CO₂ growth | `co2_growth_abs`, `co2_growth_prct` | COVID & policy shocks visible; China surged 2000–2012 |
| | A3 — CO₂ vs CO₂ incl. LUC | `co2`, `co2_including_luc` | Brazil/Indonesia: true emissions far higher than official |
| | A4 — Cumulative CO₂ | `cumulative_co2`, `share_global_cumulative_co2` | USA holds 23.5% of all historical emissions |
| **B. Economy** | B1 — CO₂ per GDP trend | `co2_per_gdp` | Efficiency improved everywhere; absolute cuts are harder |
| | B2 — Production vs Consumption | `co2`, `consumption_co2`, `trade_co2` | UK/Germany consume far more carbon than they produce |
| | B3 — Trade CO₂ share trend | `trade_co2_share` | Rich nations increasing outsourcing of emissions |
| **C. Resources** | C1 — Fuel breakdown | `coal_co2`, `oil_co2`, `gas_co2`, `cement_co2`, `flaring_co2` | China & India: coal-dominant; USA: oil/gas balanced |
| | C2 — Flaring CO₂ | `flaring_co2` | Russia, USA, Iraq top flaring — regulatory failure |
| | C3 — Land-use change | `land_use_change_co2` | Brazil's Amazon policy worked; DRC is the new frontier |
| | C4 — Energy grid intensity | `co2_per_unit_energy` | France/Brazil: clean grids; India/South Africa: coal-heavy |
| | C5 — Energy vs CO₂ | `primary_energy_consumption`, `co2` | Clean-grid countries decouple energy use from emissions |
| **D. Policy** | D1 — Global trend + milestones | `co2` (World) | Emissions rose through every climate agreement |
| | D2 — Temperature attribution | `temperature_change_from_co2`, `share_of_temperature_change_from_ghg` | USA caused most warming despite China's recent surge |
| | D3 — Full GHG breakdown | `co2`, `methane`, `nitrous_oxide` | Brazil's methane rivals its CO₂ |
| | D4 — Share of global CO₂ | `share_global_co2` | China overtook USA in 2000s; India's share growing |
| | D5 — Historical vs current | `cumulative_co2`, `co2_per_capita` | Core climate justice chart — CBDR-RC in one graph |
| | D6 — World choropleth | `co2_per_capita` | Gulf states + Australia lead per-person emissions |
| | D7 — GHG per capita heatmap | `ghg_per_capita` | UK/Germany improved; USA/Australia stubbornly high |
| **E. Synthesis** | E1 — Decoupling check | `gdp`, `co2` | Germany/UK: real decoupling; China: relative only |
| | E2 — Methane vs CO₂ | `co2`, `methane`, `nitrous_oxide` | Agriculture-heavy nations need non-CO₂ policies too |